# Final model: test evaluation, residual analysis, saved pipeline
Run this **only after** notebooks 01-03 are finished and the best model has been chosen from the cross-validation results.
The test set is evaluated **once**, here.

Selected model: **Linear Regression** (lowest CV RMSE in notebooks 01-03; confirm after rerunning them).

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

DATA_PATH = next(
    p for p in [Path("data/original/calorie_dataset.csv"),
                Path("../data/original/calorie_dataset.csv"),
                Path("calorie_dataset.csv")]
    if p.exists()
)
df = pd.read_csv(DATA_PATH)

FEATURES = ["protein_g", "carbs_g", "fat_g", "fiber_g", "sugar_g"]
TARGET = "calories"        

X = df[FEATURES]
y = df[TARGET]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "| Test:", X_test.shape, "(test set is NOT used in this notebook)")

In [ ]:
preprocessor = ColumnTransformer(transformers=[("num", StandardScaler(), FEATURES)])
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

SCORING = {
    "rmse": "neg_root_mean_squared_error",
    "mae":  "neg_mean_absolute_error",
    "r2":   "r2",
}

def summarize_cv(model, name):
    """5-fold CV on the TRAINING set only; prints mean +/- std for RMSE, MAE, R2."""
    res = cross_validate(model, X_train, y_train, cv=cv, scoring=SCORING)
    rmse, mae, r2 = -res["test_rmse"], -res["test_mae"], res["test_r2"]
    print(name)
    print(f"CV RMSE: {rmse.mean():.4f} +/- {rmse.std():.4f}")
    print(f"CV MAE : {mae.mean():.4f} +/- {mae.std():.4f}")
    print(f"CV R2  : {r2.mean():.4f} +/- {r2.std():.4f}")
    return {"model": name, "rmse_mean": rmse.mean(), "rmse_std": rmse.std(),
            "mae_mean": mae.mean(), "mae_std": mae.std(),
            "r2_mean": r2.mean(), "r2_std": r2.std()}

In [ ]:
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_style("whitegrid")
OUT_ROOT = Path("..") if Path("../data").exists() else Path(".")

final_pipe = Pipeline([
    ("pre", preprocessor),
    ("model", LinearRegression())
])
_ = summarize_cv(final_pipe, "Linear Regression (validation, for reference)")

## Final test evaluation (once)

In [ ]:
final_pipe.fit(X_train, y_train)
y_pred = final_pipe.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_mae = mean_absolute_error(y_test, y_pred)
test_r2 = r2_score(y_test, y_pred)

print("Linear Regression - TEST metrics")
print(f"RMSE: {test_rmse:.4f}")
print(f"MAE : {test_mae:.4f}")
print(f"R2  : {test_r2:.4f}")

## Residual analysis and error distribution

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

axes[0].scatter(y_test, y_pred, alpha=0.5)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, "r--")
axes[0].set_xlabel("Actual calories (kcal)")
axes[0].set_ylabel("Predicted calories (kcal)")
axes[0].set_title("Predicted vs actual")

axes[1].scatter(y_pred, residuals, alpha=0.5)
axes[1].axhline(0, color="r", linestyle="--")
axes[1].set_xlabel("Predicted calories (kcal)")
axes[1].set_ylabel("Residual (kcal)")
axes[1].set_title("Residuals vs predicted")

sns.histplot(residuals, kde=True, ax=axes[2])
axes[2].set_xlabel("Residual (kcal)")
axes[2].set_title("Error distribution")

plt.tight_layout()
plt.show()

print(f"Mean residual: {residuals.mean():.2f} kcal | Std: {residuals.std():.2f} kcal")
print(f"Largest absolute error: {residuals.abs().max():.2f} kcal")

**Finding:** _(are residuals centered on zero and evenly spread? any pattern or heteroscedasticity? where are the largest errors?)_

## Coefficients (interpretation)

In [ ]:
coefs = pd.Series(final_pipe.named_steps["model"].coef_, index=FEATURES)
print("Coefficients per 1 SD of each (standardized) feature, in kcal:")
print(coefs.sort_values(ascending=False))

## Save the full pipeline (preprocessing + model) for the Streamlit app

In [ ]:
(OUT_ROOT / "models").mkdir(exist_ok=True)
MODEL_PATH = OUT_ROOT / "models" / "calorie_best_pipeline.joblib"
joblib.dump(final_pipe, MODEL_PATH)

# sanity check: reload and confirm identical predictions
reloaded = joblib.load(MODEL_PATH)
assert np.allclose(reloaded.predict(X_test), y_pred)
print("Saved and verified:", MODEL_PATH)